# LLM Jailbreak Framework — Interactive Notebook

Use this notebook to run experiments, explore results, and generate visualizations.

> **Note:** Run from the project root directory.

In [ ]:
import sys, json
sys.path.insert(0, '..')

from utils.config import Config
from utils.logger import JBFLogger, AttackRecordLogger
from utils.reporter import Reporter
from core.generator import PromptGenerator
from core.attacker import AttackEngine
from core.evaluator import Evaluator
from core.scorer import Scorer
from models.local_model import LocalModel
from models.openai_model import OpenAIModel

print('Imports OK')

## Configuration

In [ ]:
cfg = Config('../config.yaml')
JBFLogger(log_level='WARNING')
print('Config loaded:', cfg.summarize())

## Choose a Model
Use mock mode for testing without an API key.

In [ ]:
model = LocalModel(mode='mock', mock_success_rate=0.4)

print(f'Model: {model.model_name}')
print(f'Connection OK: {model.test_connection()}')

## Run a Quick Experiment

In [ ]:
STRATEGIES = ['roleplay', 'instruction_override', 'encoding_attack']

generator  = PromptGenerator(strategies=STRATEGIES, prompts_file='../data/prompts.json', seed=42)
attacker   = AttackEngine(model, rate_limit_delay=0.0)
evaluator  = Evaluator(mode='keyword')
scorer     = Scorer()

all_results = []

for strat in STRATEGIES:
    batch = generator.generate_batch(strat, count=3)
    attacked = attacker.send_batch(batch)
    evaluated = evaluator.evaluate_batch(attacked)
    scored = scorer.score_batch(evaluated)
    all_results.extend(scored)
    print(f'{strat}: {len(scored)} results')

print(f'\nTotal results: {len(all_results)}')

## Aggregate Statistics

In [ ]:
import pandas as pd
stats = scorer.aggregate_stats(all_results)
print(json.dumps(stats, indent=2))

## Results as DataFrame

In [ ]:
df = pd.DataFrame(all_results)[[
    'strategy', 'category', 'success', 'eval_confidence',
    'risk_level', 'risk_score', 'eval_method'
]]
df

## Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='darkgrid')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Jailbreak Experiment Results', fontsize=14)

sr = df.groupby('strategy')['success'].mean()
sr.plot(kind='bar', ax=axes[0], color='coral', edgecolor='black')
axes[0].set_title('Success Rate by Strategy')
axes[0].set_ylabel('Success Rate')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=30)

df['risk_score'].hist(ax=axes[1], bins=10, color='steelblue', edgecolor='black')
axes[1].set_title('Risk Score Distribution')
axes[1].set_xlabel('Risk Score')

df['risk_level'].value_counts().plot(kind='bar', ax=axes[2], color='mediumpurple', edgecolor='black')
axes[2].set_title('Risk Level Counts')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../reports/notebook_chart.png', dpi=120, bbox_inches='tight')
plt.show()

## Save Results

In [ ]:
record_logger = AttackRecordLogger('../outputs/notebook_results.json')
record_logger.save_batch(all_results)

reporter = Reporter('../reports/')
reporter.save_markdown_report(all_results, stats)
print(f'Saved {record_logger.record_count} records.')